# Week 10: Spaceports and orbits (Capstone 2 week)

**Track:** Orbital Analyst (Intermediate)  
**Full primer + quiz:** [https://launchdetect.com/academy/week/10/](https://launchdetect.com/academy/week/10/)  
**Track index:** [https://launchdetect.com/academy/orbital-analyst/](https://launchdetect.com/academy/orbital-analyst/)

---

_Track 2 culminates here: combine ground station coverage analysis with orbital mechanics to answer the matching question — given an orbit, which spaceport? Given a spaceport, which orbits? The capstone delivers a ground-track coverage tool._


## Why this week matters

Week 10 is the capstone of Track 2. You're combining everything: orbital mechanics (Weeks 7-8), ground-station visibility (Week 9), spatial analysis (Week 5), and PostGIS-shaped thinking (Week 6) into a single tool.

The deliverable — a function that takes any TLE and emits ground track + coverage polygon + country-overflight dwell table — is a real operational artifact. Satellite operators use exactly this kind of tool for mission planning, customers use it for tasking, and journalists use it to verify claims like 'this satellite passed over X'.

**Why this combination?** Because each piece in isolation isn't useful, but together they answer the question 'where is this satellite operationally relevant?' with one CSV-shaped answer. Ground track says where; coverage says how much; country dwell says under whose jurisdiction. Three layers, one map, one decision.


## Learning objectives

By the end of this lab you will be able to:

- Match spaceport latitude to feasible orbital inclinations
- Explain why Kourou is the GEO launch capital
- Identify which spaceports can serve sun-synchronous orbits
- Build a coverage polygon for a hypothetical 1000-km swath sensor


## Setup — and why these dependencies

Continues with `skyfield` + `pyproj.Geod` + `leafmap`. No new deps.


In [ ]:
# Install required packages
!pip install -q leafmap[common] skyfield geopandas shapely


## The approach (and why)

Propagate ISS for 24h, build ground track from sub-satellite points, build 1000-km swath coverage as geodesic-buffered points, and (in the capstone — your work) join against Natural Earth admin-0 country polygons to compute per-country dwell.

**Why 1000-km swath specifically?** It's the rough field-of-view of a wide-area imaging satellite (Landsat is 185 km, Sentinel-2 is 290 km, Sentinel-1 IW is 250 km, 1000 km is closer to wide-field altimetry or SAR strip mode). Pick the number that matches your real sensor in production.

**Why country dwell is the most operationally-useful output?** Because legal and regulatory regimes change at national borders. A satellite that overflies country X for 3 minutes/day per pass might need export-control documentation that one overflying for 30 seconds/year doesn't.


In [ ]:
# Week 10: ground-track coverage tool — CAPSTONE 2 START.
import leafmap.foliumap as leafmap
from skyfield.api import EarthSatellite, load, wgs84
from pyproj import Geod

tle1 = "1 25544U 98067A   24130.50145833  .00018539  00000-0  33188-3 0  9994"
tle2 = "2 25544  51.6406 348.5395 0006703 117.9568 358.1729 15.50289267449420"
ts = load.timescale()
iss = EarthSatellite(tle1, tle2, "ISS", ts)

# 24h ground track at 1 min
t0 = ts.now()
points = [(wgs84.subpoint(iss.at(ts.tt_jd(t0.tt + i/1440))).longitude.degrees,
           wgs84.subpoint(iss.at(ts.tt_jd(t0.tt + i/1440))).latitude.degrees)
          for i in range(0, 1440, 5)]  # every 5 min

# 1000-km-swath buffer around each point (approximate, geodesic)
geod = Geod(ellps="WGS84")
swath_pts = []
for lon, lat in points[::6]:  # every 30 min for clarity
    ring = [geod.fwd(lon, lat, az, 500_000)[:2] for az in range(0, 361, 30)]
    swath_pts.extend(ring)

m = leafmap.Map(center=[0, 0], zoom=2)
m.add_geojson({"type":"Feature","geometry":{"type":"LineString","coordinates":points}},
              style={"color":"#4a9eff","weight":2})
# (Coverage polygon visualization left as a TODO — see capstone rubric)
m

# TODO: compute (1) ground track as GeoJSON, (2) 1000-km coverage POLYGON via
# unary_union of per-vertex buffers, (3) country-overflight dwell table via
# Natural Earth admin-0 spatial join. CAPSTONE 2 deliverable.


## What just happened — and why it works

The ground track is built the same way as Week 8 (sub-satellite points, antimeridian split). The coverage swath is a series of geodesic-distance rings around the ground track — for each track vertex, walk 500 km in azimuths 0°-360° to build the swath boundary at that moment.

Stitching the per-vertex rings into a single coverage polygon (in the capstone) is a `shapely.ops.unary_union` of all the buffered rings — a one-liner that produces the swept-area polygon. From there, intersecting with country polygons gives you the per-country area + dwell time table.

Notice how each previous-week concept slots in: Week 5's geodesic buffering is the swath ring; Week 6's `ST_DWithin` is the country-join logic (if you push to PostGIS); Week 8's ground track is the input. **This is what synthesis weeks look like.**


## Common gotchas

- **Coverage polygons can self-intersect.** A satellite on a highly inclined orbit swaths the same patch of ground twice per orbit. The merged polygon needs to tolerate this — `shapely.unary_union` handles it.
- **Antimeridian crossings break naive area calculations.** Use geography-mode PostGIS or `pyproj.Geod.geometry_area_perimeter()` to get correct global area.
- **Per-country dwell can exceed total propagation time** if a satellite overflies both halves of a country (e.g., Russia, USA across multiple passes). Sum per-pass dwell, not per-vertex.
- **Polygon validity matters in PostGIS.** Run `ST_MakeValid(geom)` before any spatial join if you've built polygons programmatically — invalid geometries error out the join silently.


## Self-check

Verify your solution against these checks before considering the lab complete:

- [ ] Output type matches the expected format (GeoJSON / PNG / table / etc.).
- [ ] No exceptions raised on a clean run.
- [ ] Result is visually plausible — open the map cell, scan the values, sanity-check magnitudes.
- [ ] Quiz on the [week page](https://launchdetect.com/academy/week/10/) — try answering before checking the key.

---

Found a bug or want to contribute an improvement? Open an issue or PR at [github.com/launchdetect/academy-labs](https://github.com/launchdetect/academy-labs).
